# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [1]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"

df = pd.read_csv(url)

# Standardize column names: lowercase, no spaces, no dots
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_", regex=False)
    .str.replace(".", "", regex=False)
)

df.head()

,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2/18/11,Employed,M,...,0.0,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,NaN
1,1,KX64629,California,2228.525238,No,Basic,College,1/18/11,Unemployed,F,...,0.0,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,NaN
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2/10/11,Employed,M,...,0.0,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,1/17/11,Medical Leave,F,...,NaN,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,NaN


In [2]:
# Quick overview
df.shape, df.columns.tolist()

((10910, 26),
 ['unnamed:_0',
  'customer',
  'state',
  'customer_lifetime_value',
  'response',
  'coverage',
  'education',
  'effective_to_date',
  'employmentstatus',
  'gender',
  'income',
  'location_code',
  'marital_status',
  'monthly_premium_auto',
  'months_since_last_claim',
  'months_since_policy_inception',
  'number_of_open_complaints',
  'number_of_policies',
  'policy_type',
  'policy',
  'renew_offer_type',
  'sales_channel',
  'total_claim_amount',
  'vehicle_class',
  'vehicle_size',
  'vehicle_type'])

In [3]:
# Check missing values and duplicated rows
display(df.isna().sum().sort_values(ascending=False))
print("Duplicated rows:", df.duplicated().sum())

vehicle_type                     5482
number_of_open_complaints         633
months_since_last_claim           633
response                          631
state                             631
vehicle_class                     622
vehicle_size                      622
customer                            0
customer_lifetime_value             0
unnamed:_0                          0
gender                              0
employmentstatus                    0
effective_to_date                   0
education                           0
coverage                            0
monthly_premium_auto                0
income                              0
location_code                       0
number_of_policies                  0
months_since_policy_inception       0
marital_status                      0
policy_type                         0
sales_channel                       0
renew_offer_type                    0
policy                              0
total_claim_amount                  0
dtype: int64

Duplicated rows: 0


In [4]:
# Convert date column to datetime
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], errors="coerce")

# Create month column for later analysis
df["month"] = df["effective_to_date"].dt.month_name()

# Optional: order months chronologically when displayed
month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

df[["effective_to_date", "month"]].head()

C:\Users\estef\AppData\Local\Temp\ipykernel_32372\898548467.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], errors="coerce")


,effective_to_date,month
0,2011-02-18,February
1,2011-01-18,January
2,2011-02-10,February
3,2011-01-11,January
4,2011-01-17,January


## 1. Customers with low total claim amount and positive response

In [5]:
low_claim_yes_response = df[
    (df["total_claim_amount"] < 1000) &
    (df["response"] == "Yes")
].copy()

low_claim_yes_response.head()

,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,...,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,January
8,8,FM55990,California,5989.773931,Yes,Premium,College,2011-01-19,Employed,M,...,1,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,NaN,January
15,15,CW49887,California,4626.801093,Yes,Basic,Master,2011-01-16,Employed,F,...,1,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,NaN,January
19,19,NJ54277,California,3746.751625,Yes,Extended,College,2011-02-26,Employed,F,...,1,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,A,February
27,27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2011-02-28,Employed,F,...,1,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,NaN,February


In [6]:
low_claim_yes_response.shape

(1399, 27)

## 2. Average monthly premium and customer lifetime value by policy type and gender

In [7]:
yes_customers = df[df["response"] == "Yes"].copy()

policy_gender_analysis = (
    yes_customers
    .groupby(["policy_type", "gender"])
    .agg(
        customers=("customer", "count"),
        avg_monthly_premium=("monthly_premium_auto", "mean"),
        avg_customer_lifetime_value=("customer_lifetime_value", "mean"),
        avg_total_claim_amount=("total_claim_amount", "mean"),
        median_total_claim_amount=("total_claim_amount", "median")
    )
    .round(2)
    .reset_index()
)

policy_gender_analysis

,policy_type,gender,customers,avg_monthly_premium,avg_customer_lifetime_value,avg_total_claim_amount,median_total_claim_amount
0,Corporate Auto,F,169,94.30,7712.63,433.74,420.04
1,Corporate Auto,M,154,92.19,7944.47,408.58,364.80
2,Personal Auto,F,540,99.00,8339.79,452.97,424.33
3,Personal Auto,M,536,91.09,7448.38,457.01,412.80
4,Special Auto,F,35,92.31,7691.58,453.28,420.04
5,Special Auto,M,32,86.34,8247.09,429.53,345.60


In [8]:
# higher CLV and lower claims could be seen as more attractive
policy_gender_analysis["clv_minus_avg_claim"] = (
    policy_gender_analysis["avg_customer_lifetime_value"] -
    policy_gender_analysis["avg_total_claim_amount"]
).round(2)

policy_gender_analysis.sort_values("clv_minus_avg_claim", ascending=False)

,policy_type,gender,customers,avg_monthly_premium,avg_customer_lifetime_value,avg_total_claim_amount,median_total_claim_amount,clv_minus_avg_claim
2,Personal Auto,F,540,99.00,8339.79,452.97,424.33,7886.82
5,Special Auto,M,32,86.34,8247.09,429.53,345.60,7817.56
1,Corporate Auto,M,154,92.19,7944.47,408.58,364.80,7535.89
0,Corporate Auto,F,169,94.30,7712.63,433.74,420.04,7278.89
4,Special Auto,F,35,92.31,7691.58,453.28,420.04,7238.30
3,Personal Auto,M,536,91.09,7448.38,457.01,412.80,6991.37


## 3. Number of customers by state, filtered to states with more than 500 customers

In [9]:
customers_by_state = (
    df.groupby("state")
    .size()
    .reset_index(name="number_of_customers")
    .sort_values("number_of_customers", ascending=False)
)

customers_by_state

,state,number_of_customers
1,California,3552
3,Oregon,2909
0,Arizona,1937
2,Nevada,993
4,Washington,888


In [10]:
states_more_than_500 = customers_by_state[
    customers_by_state["number_of_customers"] > 500
]

states_more_than_500

,state,number_of_customers
1,California,3552
3,Oregon,2909
0,Arizona,1937
2,Nevada,993
4,Washington,888


## 4. Maximum, minimum and median customer lifetime value by education level and gender

In [11]:
clv_by_education_gender = (
    df.groupby(["education", "gender"])["customer_lifetime_value"]
    .agg(["max", "min", "median"])
    .round(2)
    .reset_index()
)

clv_by_education_gender

,education,gender,max,min,median
0,Bachelor,F,73225.96,1904.00,5640.51
1,Bachelor,M,67907.27,1898.01,5548.03
2,College,F,61850.19,1898.68,5623.61
3,College,M,61134.68,1918.12,6005.85
4,Doctor,F,44856.11,2395.57,5332.46
5,Doctor,M,32677.34,2267.60,5577.67
6,High School or Below,F,55277.45,2144.92,6039.55
7,High School or Below,M,83325.38,1940.98,6286.73
8,Master,F,51016.07,2417.78,5729.86
9,Master,M,50568.26,2272.31,5579.10


In [12]:
# Sort by median CLV to identify stronger groups
clv_by_education_gender.sort_values("median", ascending=False)

,education,gender,max,min,median
7,High School or Below,M,83325.38,1940.98,6286.73
6,High School or Below,F,55277.45,2144.92,6039.55
3,College,M,61134.68,1918.12,6005.85
8,Master,F,51016.07,2417.78,5729.86
0,Bachelor,F,73225.96,1904.00,5640.51
2,College,F,61850.19,1898.68,5623.61
9,Master,M,50568.26,2272.31,5579.10
5,Doctor,M,32677.34,2267.60,5577.67
1,Bachelor,M,67907.27,1898.01,5548.03
4,Doctor,F,44856.11,2395.57,5332.46


## Bonus 5. Policies sold by state and month

In [13]:
policies_by_state_month = pd.pivot_table(
    df,
    index="state",
    columns="month",
    values="customer",
    aggfunc="count",
    fill_value=0
)

# Reorder columns chronologically if all months are present
policies_by_state_month = policies_by_state_month.reindex(
    columns=[m for m in month_order if m in policies_by_state_month.columns]
)

policies_by_state_month

month,January,February
state,,
Arizona,1008,929
California,1918,1634
Nevada,551,442
Oregon,1565,1344
Washington,463,425


## Bonus 6. Policies sold by month for the top 3 states

In [15]:
top_3_states = (
    df.groupby("state")
    .size()
    .sort_values(ascending=False)
    .head(3)
    .index
)

top_3_states

Index(['California', 'Oregon', 'Arizona'], dtype='str', name='state')

In [16]:
top_3_states_df = df[df["state"].isin(top_3_states)].copy()

top_3_state_month_table = pd.pivot_table(
    top_3_states_df,
    index="state",
    columns="month",
    values="customer",
    aggfunc="count",
    fill_value=0
)

top_3_state_month_table = top_3_state_month_table.reindex(
    columns=[m for m in month_order if m in top_3_state_month_table.columns]
)

top_3_state_month_table

month,January,February
state,,
Arizona,1008,929
California,1918,1634
Oregon,1565,1344


## Bonus 7. Response rate by marketing channel

In [17]:
response_by_channel = (
    df.groupby("sales_channel")
    .agg(
        total_customers=("customer", "count"),
        yes_responses=("response", lambda x: (x == "Yes").sum())
    )
    .reset_index()
)

response_by_channel["response_rate"] = (
    response_by_channel["yes_responses"] / response_by_channel["total_customers"] * 100
).round(2)

response_by_channel.sort_values("response_rate", ascending=False)

,sales_channel,total_customers,yes_responses,response_rate
0,Agent,4121,742,18.01
3,Web,1626,177,10.89
1,Branch,3022,326,10.79
2,Call Center,2141,221,10.32
